# LLM Evaluation with evidently.ai


Evidently Python library: https://github.com/evidentlyai/evidently. Docs: https://docs.evidentlyai.com

In [56]:
#!pip install evidently[llm]

[No output generated]

**Attribution & License**

This notebook is adapted from: <evidentlyai/community-examples> (<https://github.com/evidentlyai/community-examples.git>), licensed under the
Apache License, Version 2.0. © Original authors.

Modifications: <models and paths to models> by <Simeon Harrison/EuroCC Austria>, © 2025.

# Tutorial structure

In this tutorial, we'll walk through:
- **Basics**: anatomy of a single eval.
- **Reference-based evaluations**: when you have a ground truth to compare against.
  - Deterministic matching
  - Semantic similarity and BERTScore
  - LLM-as-a-judge (Correctness)
- **Open-ended evaluations**: when there is no “correct” answer.
  - Regular expressions
  - Text statistics
  - Custom Python check
  - Semantic similarity
  - ML model scoring
  - LLM-as-a-judge (Faithfulness, Completeness)
- **Multi-turn evals**: when you are evaluating a complete conversation session, not just a single input-output pair.

We'll use a mini-dataset for a financial assistant chatbot.

# Imports

In [57]:
import pandas as pd
import os
from evidently import Report
from evidently import Dataset, DataDefinition
from evidently.descriptors import TextLength, Sentiment, HuggingFace, IncludesWords, SemanticSimilarity, ExactMatch, BERTScore, SentenceCount
from evidently.descriptors import LLMEval, PIILLMEval, DeclineLLMEval, CorrectnessLLMEval, FaithfulnessLLMEval
from evidently.descriptors import ColumnTest, TestSummary, CustomColumnDescriptor
from evidently.llm.templates import BinaryClassificationPromptTemplate, MulticlassClassificationPromptTemplate
from evidently.core.datasets import DatasetColumn
from evidently.presets import TextEvals
from evidently.tests import eq, gte, lte
from evidently.ui.workspace import CloudWorkspace

[No output generated]

In [58]:
# === GPU & Model Status Check ===
import gc
import torch

print("=== Initial Resource Status ===")

# GPU Status - Use pynvml for SYSTEM-WIDE memory (not just this process)
try:
    import pynvml
    pynvml.nvmlInit()
    device_count = pynvml.nvmlDeviceGetCount()
    print(f"\nGPU Count: {device_count}")

    for i in range(device_count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        name = pynvml.nvmlDeviceGetName(handle)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)

        total_gb = info.total / 1024**3
        used_gb = info.used / 1024**3
        free_gb = info.free / 1024**3
        usage_pct = (info.used / info.total) * 100

        print(f"\nGPU {i}: {name}")
        print(f"  Total:  {total_gb:.2f} GB")
        print(f"  Used:   {used_gb:.2f} GB ({usage_pct:.1f}%)")
        print(f"  Free:   {free_gb:.2f} GB")

        # Warning if low on memory (7B model needs ~5GB with 4-bit quantization)
        if free_gb < 6.0:
            print(f"  ⚠️  WARNING: Low GPU memory! Model loading may fail.")
            print(f"      Consider running cleanup cells in other notebooks first.")

    pynvml.nvmlShutdown()
except ImportError:
    print("\n⚠️  pynvml not installed - falling back to PyTorch (per-process only)")
    if torch.cuda.is_available():
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        for i in range(torch.cuda.device_count()):
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            allocated = torch.cuda.memory_allocated(i) / 1024**3
            print(f"  GPU {i}: {allocated:.2f} / {total:.2f} GB (THIS PROCESS ONLY)")
    else:
        print("No GPU available - using CPU")
except Exception as e:
    print(f"\nGPU status check failed: {e}")

# Ollama Status
try:
    import ollama
    running = ollama.ps()
    models = running.get("models", [])
    if models:
        print("\nOllama models currently loaded:")
        for m in models:
            name = m.get("name", "Unknown")
            vram = m.get("size_vram", 0) / 1024**3
            print(f"  - {name} ({vram:.2f} GB VRAM)")
    else:
        print("\nNo Ollama models currently loaded")
except Exception as e:
    print(f"\nOllama status check skipped: {e}")

print("\n" + "="*40)

=== Initial Resource Status ===

GPU Count: 1

GPU 0: b'NVIDIA GeForce RTX 4080 SUPER'
  Total:  15.99 GB
  Used:   9.06 GB (56.7%)
  Free:   6.93 GB

No Ollama models currently loaded



# Part 1. A very basic example

In [59]:
data = [
    ["What is the capital of France?", "The capital of France is Paris."],
    ["Can penguins fly?", "No, penguins cannot fly but they are excellent swimmers."],
    ["Help me with homework?", "I'm here to help guide you, but I can't do your homework for you."],
    ["Is water wet?", "Yes, water is considered wet because it makes things wet."],
    ["Do fish sleep?", "Yes, fish do sleep, though not in the same way humans do."],
    ["What is 2 + 2?", "2 + 2 equals 4."],
    ["Is the Earth flat?", "No, the Earth is a sphere."],
    ["Can dogs talk?", "Dogs can't talk in human language, but they communicate through barks, body language, and behavior. For example, a wagging tail often means happiness, while growling can signal fear or aggression. They're great at expressing themselves without words."],
    ["What's your name?", "I’m a virtual assistant bot."],
    ["Are bananas berries?", "Yes, botanically speaking, bananas are classified as berries."]
]

columns = ["question", "answer"]

eval_data = pd.DataFrame(data, columns=columns)

[No output generated]

In [60]:
pd.set_option('display.max_colwidth', None)

[No output generated]

In [61]:
eval_data.head()

                         question  \
0  What is the capital of France?   
1               Can penguins fly?   
2          Help me with homework?   
3                   Is water wet?   
4                  Do fish sleep?   

                                                              answer  
0                                    The capital of France is Paris.  
1           No, penguins cannot fly but they are excellent swimmers.  
2  I'm here to help guide you, but I can't do your homework for you.  
3          Yes, water is considered wet because it makes things wet.  
4          Yes, fish do sleep, though not in the same way humans do.  

Prepare the Dataset for Evidently to work with:

In [62]:
definition = DataDefinition(text_columns=["question", "answer"])

[No output generated]

In [63]:
eval_df = Dataset.from_pandas(
    pd.DataFrame(eval_data),
    data_definition=definition)

[No output generated]

A **Descriptor** is a row-level score or label that assesses a specific quality of a given text. It’s different from dataset metrics (like accuracy or precision) that give a score for an entire dataset. Each Descriptor returns a result that can be:
- Numerical. Any scores like symbol count or sentiment score.
- Categorical. Labels or binary “true”/“false” results for pattern matches.
- Text string. Like explanations generated by LLM.

 Let's use text length for illustration:

In [64]:
eval_df.add_descriptors(descriptors=[
    TextLength("answer", alias="Answer Length"),
])

[No output generated]

In [65]:
eval_df.as_dataframe()

                         question  \
0  What is the capital of France?   
1               Can penguins fly?   
2          Help me with homework?   
3                   Is water wet?   
4                  Do fish sleep?   
5                  What is 2 + 2?   
6              Is the Earth flat?   
7                  Can dogs talk?   
8               What's your name?   
9            Are bananas berries?   

                                                                                                                                                                                                                                                       answer  \
0                                                                                                                                                                                                                             The capital of France is Paris.   
1                                                                             

See all implemented Descriptors: https://docs.evidentlyai.com/metrics/all_descriptors

**Descriptor tests**. You can also add conditional tests for pass/fail for each row.

In [66]:
# You can also create the dataframe together with adding the descriptors.

eval_df = Dataset.from_pandas(
    pd.DataFrame(eval_data),
    data_definition=definition,
    descriptors=[TextLength("answer", alias="Answer Length",
                            tests=[gte(100, alias="Answer is too long")])])

eval_df.as_dataframe()

                         question  \
0  What is the capital of France?   
1               Can penguins fly?   
2          Help me with homework?   
3                   Is water wet?   
4                  Do fish sleep?   
5                  What is 2 + 2?   
6              Is the Earth flat?   
7                  Can dogs talk?   
8               What's your name?   
9            Are bananas berries?   

                                                                                                                                                                                                                                                       answer  \
0                                                                                                                                                                                                                             The capital of France is Paris.   
1                                                                             

**Report**. Summarize evaluation results.

In [67]:
report = Report([
    TextEvals()
])

my_eval = report.run(eval_df)

[No output generated]

In [68]:
my_eval

Other output formats:

In [69]:
# my_eval.json()
# my_eval.dict()
# my_eval.save_html("file.html")

[No output generated]

**Upload to Evidently Cloud**. Connect to Cloud:

In [70]:
# import os, getpass
# os.environ["EVIDENTLY_API_KEY"] = getpass.getpass("Evidently API key (hidden): ")

[No output generated]

In [71]:
# ws = CloudWorkspace(
#     token=os.getenv("EVIDENTLY_API_KEY"),
#     url="https://app.evidently.cloud",
# )

[No output generated]

Create or connect to a Project.

In [72]:
# project = ws.get_project("YOUR_PROJECT_ID")

[No output generated]

In [73]:
# project = ws.create_project("My project name", org_id="YOUR_ORG_ID")
# project.description = "My project description"
# project.save()

[No output generated]

Upload:

In [74]:
# ws.add_run(project.id, my_eval, include_data=True)

[No output generated]

# Part 2. Reference-based evals

Generate a toy dataset. Let's imagine a Q&A (RAG) use case where the system generates the response based on the retrieved context.

In [75]:
eval_data = pd.DataFrame([
    {
        "question": "Will my transaction go through if I don't have enough funds?",
        "context": "Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.",
        "answer": "Yes, your transaction will go through if overdraft protection is enabled, but a $35 fee will apply",
        "reference_answer": "Yes, with overdraft protection, your transaction will complete, but you will be charged $35."
    },
    {
        "question": "How do I block my card if it's lost?",
        "context": "To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.",
        "answer": "Go to the Cards section, select your card, and tap 'Block card' to block it instantly.",
        "reference_answer": "Open the app, go to Cards, choose your card, and tap 'Block card'. Blocking is immediate."
    },
    {
        "question": "Do you offer loans in Argentina?",
        "context": "FinBot currently offers loans in 20+ locations, including the US, Canada, and selected EU countries.",
        "answer": "Yes, FinBot offers personal loans in Argentina with competitive rates.",  # Incorrect
        "reference_answer": "No, FinBot does not currently offer loans in Argentina."
    },
    {
        "question": "Is there a fee for using an ATM in Mexico?",
        "context": "ATM withdrawals in Mexico are free when using partner ATMs. Non-partner ATMs incur a $2.50 fee per withdrawal, which is deducted immediately.",
        "answer": "You'll be charged $2.50.",
        "reference_answer": "Yes, the fee is $2.50 for non-partner ATMs. Partner ATMs are free."
    },
    {
        "question": "Can I cancel a transaction after it's sent?",
        "context": "Outgoing transactions cannot be canceled once processed. Users may initiate a recall request, but success is not guaranteed. The recipient's bank must agree to reverse the transfer.",
        "answer": "I am afraid I do not have information to answer this question.",
        "reference_answer": "No, but you can submit a recall request. It depends on the recipient's bank."
    }
])

[No output generated]

## Ground truth (Mocked)

Let's first take a look at the starting point: a golden dataset of expected questions and answers.

In [76]:
golden_df = eval_data[["question", "reference_answer"]].copy()
golden_df.head()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                               reference_answer  
0  Yes, with overdraft protection, your transaction will complete, but you will be charged $35.  
1     Open the app, go to Cards, choose your card, and tap 'Block card'. Blocking is immediate.  
2                                       No, FinBot does not currently offer loans in Argentina.  
3                            Yes, the fee is $2.50 for non-partner ATMs. Partner ATMs are free.  
4                  No, but you can submit a recall request. It depends on the recipient's bank.  

In [77]:
golden_df.size

10

## Scored data (Mocked)

Let's assume we ran it through our app and got the actual answer and context used to generate it.

In [78]:
eval_data.head()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

# Reference-based evals

## Deterministic

Exact match - let's use it for illustration.

In [79]:
definition = DataDefinition(text_columns=["question", "context", "answer", "reference_answer"])

[No output generated]

In [80]:
eval_df = Dataset.from_pandas(
    pd.DataFrame(eval_data),
    data_definition=definition,
    descriptors=[ExactMatch(columns=["answer", "reference_answer"], alias="ExactMatch")])

[No output generated]

In [81]:
eval_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

Exact Match checks if the generated response matches the reference text exactly.

However, in real-world LLM output, even perfectly valid answers may use different wording or structure. This method is too strict.

## Semantic match

Let's compare semantic match.

We’ll use two approaches:

*   **SemanticSimilarity**: cosine similarity over sentence embeddings. This method produces a single vector per sentence using a built-in embedding model. Measures closeness in meaning between answer and reference. Outputs a float between 0 and 1, where 0 is opposite meanings, 0.5 is unrelated, and 1 is exactly matching.
*   **BERTScore** evaluates token-level similarity using contextual embeddings from BERT. It aligns each token in the candidate sentence with the most similar token in the reference sentence based on cosine similarity of their embeddings. Precision, recall, and F1 scores are calculated over these alignments, with the F1 score used as the final metric.

In [82]:
eval_df.add_descriptors(descriptors=[
    SemanticSimilarity(columns=["answer", "reference_answer"], alias="Semantic Similarity"),
    BERTScore(columns=["answer", "reference_answer"], alias="BERTScore"),
])

[No output generated]

In [83]:
eval_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

While embedding-based metrics are helpful for measuring overall semantic closeness (and help us capture issues like a denial to respond), they aren't always precise enough for factual evaluations. These methods rely on vector similarity, so they may consider two responses "similar" even if they differ in one little detail like reversing a yes/no fact.

## LLM as a judge

We can achieve better result with LLM-based judges that can reason about meaning or detect contradictions between texts.

In [ ]:
import os
from evidently.llm.utils.wrapper import OllamaOptions

# === Ollama Configuration for evidently.ai ===
# evidently.ai uses LiteLLM which supports Ollama natively
# See: https://docs.litellm.ai/docs/providers/ollama

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
os.environ["OLLAMA_HOST"] = OLLAMA_HOST

# === Model Configuration ===
HF_LLM_MODEL = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF"
OLLAMA_LLM_MODEL = f"hf.co/{HF_LLM_MODEL}:Q4_K_M"

# Create OllamaOptions with api_url for evidently
OLLAMA_OPTIONS = OllamaOptions(api_url=OLLAMA_HOST)

print(f"Ollama host: {OLLAMA_HOST}")
print(f"Model for LLM judges: {OLLAMA_LLM_MODEL}")

In [85]:
eval_df.add_descriptors(
    descriptors=[
        CorrectnessLLMEval("answer", target_output="reference_answer", provider="ollama", model=OLLAMA_LLM_MODEL),
    ],
    options=OLLAMA_OPTIONS
)

[No output generated]

In [86]:
eval_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

Let's create a custom judge that will instead use 4 categories based on what we observe.

Let's re-import data so that we drop the existing descriptors:

In [87]:
eval_df_2 = Dataset.from_pandas(
    pd.DataFrame(eval_data),
    data_definition=definition)

[No output generated]

In [88]:
correctness_multiclass = MulticlassClassificationPromptTemplate(
    pre_messages=[("system", "You are a judge that evaluates the factual alignment of two chatbot answers.")],
    criteria="""You are given a new answer and a reference answer. Classify the new answer based on how it compares to the reference.
    ===
    Reference: {reference_answer} """,
    category_criteria={
        "fully_correct": "The answer matches the reference in all factual and semantic details.",
        "incomplete": "The answer is correct in what it says but leaves out details from the reference.",
        "adds_claims": "The answer does not contradict reference but introduces new claims not supported by the reference.",
        "contradictory": "The answer contradicts specific facts or meaning in the reference.",
    },
    uncertainty="unknown",
    include_reasoning=True,
    include_scores=False
)

[No output generated]

In [89]:
eval_df_2.add_descriptors(descriptors=[
    LLMEval("answer",
        template=correctness_multiclass,
        additional_columns={"reference_answer": "reference_answer"},
        provider="ollama",
        model=OLLAMA_LLM_MODEL,
        alias="Multi-class correctness"
    )
], options=OLLAMA_OPTIONS)

[No output generated]

In [90]:
eval_df_2.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

# Part 3. Reference-free evals

In production, or high-volume testing, you may not have a reference answer. In this case, you can run open-ended evals judging only the final generation. In many cases, you can also use supplementary information - like question and context in your evaluations.

In [91]:
prod_data = eval_data[["question", "context", "answer"]].copy()

[No output generated]

In [92]:
prod_data.head()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

In [93]:
definition = DataDefinition(text_columns=["question", "context", "answer"])

prod_df = Dataset.from_pandas(
    pd.DataFrame(prod_data),
    data_definition=definition)

[No output generated]

## Word presence

Or you can use "Contains", a custom RegEx, etc.

In [94]:
prod_df.add_descriptors(descriptors=[
     IncludesWords("answer",
              words_list=["hello", "hi", "good afternoon"],
              mode="any", alias="Says hi"),
     IncludesWords("answer",
                    words_list=["sorry", "apologies", "apologize", "cannot", "afraid"],
                    mode="any",
                    alias="Declines")
])

[No output generated]

In [95]:
prod_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

## Text stats

In [96]:
prod_df = Dataset.from_pandas(
    pd.DataFrame(prod_data),
    data_definition=definition,
    descriptors=[
        SentenceCount("answer", alias="Sentence_Count")])

[No output generated]

In [97]:
prod_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

Depending on the use case, could be `IsValidJSON()` etc.

## Custom Python check

Implement a function that takes a Pandas Series as input and return a transformed Series. For example, to check if the column is empty:

In [98]:
def is_empty(data: DatasetColumn) -> DatasetColumn:
    return DatasetColumn(
        type="cat",
        data=pd.Series([
            "EMPTY" if val == "" else "NON EMPTY"
            for val in data.data]))

[No output generated]

In [99]:
prod_df.add_descriptors(descriptors=[
    CustomColumnDescriptor("answer", is_empty, alias="is_empty"),
])

[No output generated]

In [100]:
prod_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

## Semantic similarity

You can use semantic similarity between answer and context, or answer and question as proxies for hallucinations and relevance.

In [101]:
prod_df.add_descriptors(descriptors=[
     SemanticSimilarity(columns=["answer", "context"], alias="Hallucination proxy"),
     SemanticSimilarity(columns=["answer", "question"], alias="Relevance proxy")
])

[No output generated]

In [102]:
prod_df.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

## ML models

Example: word-based sentiment model (-1 negative, 0 neutral, 1 positive).

In [103]:
prod_df_2 = Dataset.from_pandas(
    pd.DataFrame(prod_data),
    data_definition=definition,
    descriptors=[Sentiment("answer", alias="Sentiment")])

[No output generated]

In [104]:
prod_df_2.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

Example: custom model from HuggingFace. https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (Zero-shot classifier. You can provide candidate labels as params.)

In [105]:
# HuggingFace model is downloaded automatically - no local path needed
# The DeBERTa model below is used directly from HuggingFace Hub

[No output generated]

In [106]:
prod_df_2.add_descriptors(descriptors=[
   HuggingFace("answer",
               model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
               params={"labels": ["finance", "other"], "threshold":0.5},
               alias="Topic"
   )
])

Device set to use cuda:0


In [107]:
prod_df_2.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

## LLM judge

Using LLM judge to check for hallucinations (contradictions between answer and context).

In [108]:
prod_df_2 = Dataset.from_pandas(
    pd.DataFrame(prod_data),
    data_definition=definition)

[No output generated]

In [109]:
prod_df_2.add_descriptors(descriptors=[
     FaithfulnessLLMEval("answer", context="context", alias="Faithfulness", provider="ollama", model=OLLAMA_LLM_MODEL),
     TextLength("answer", alias="Length")
], options=OLLAMA_OPTIONS)

[No output generated]

In [110]:
prod_df_2.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

Let's create a custom helpfulness evaluator.

In [111]:
completeness = BinaryClassificationPromptTemplate(
    pre_messages=[("system", "You are an evaluator assessing whether a chatbot response is sufficiently complete and informative on its own.")],
    criteria = """A COMPLETE response should be a full sentence or paragraph, and be easy to understand on its own.
    For example: "Yes, you can issue additional credit card for a relative.", or longer.
    A TOO-SHORT response is overly brief or vague—for example, just a number or a simple yes/no—without additional context.
    For example: "Yes, you can."
    """,
    target_category="complete",
    non_target_category="too-short",
    uncertainty="unknown",
    include_reasoning=True
)


[No output generated]

In [112]:
prod_df_2.add_descriptors(descriptors=[
    LLMEval("answer",
        template=completeness,
        provider="ollama",
        model=OLLAMA_LLM_MODEL,
        alias="Answer completeness"
    )
], options=OLLAMA_OPTIONS)

[No output generated]

In [113]:
prod_df_2.as_dataframe()

                                                       question  \
0  Will my transaction go through if I don't have enough funds?   
1                          How do I block my card if it's lost?   
2                              Do you offer loans in Argentina?   
3                    Is there a fee for using an ATM in Mexico?   
4                   Can I cancel a transaction after it's sent?   

                                                                                                                                                                                           context  \
0       Overdraft protection allows transactions to be completed even if the account balance is insufficient. This service incurs a $35 fee per overdraft event unless linked to a backup account.   
1  To block a lost or stolen card, users should immediately navigate to the Cards section in the FinBot app, select the affected card, and tap 'Block card'. Blocking is instant and irreversible.   
2  

# Advanced: multi-turn conversation evals

For dialog systems, it can be useful to analyze multiple responses together. This allows you to assess metrics that apply to the entire session, such as to surface conversations with specific failures, like:
- Denial to respond
- Presence of PII

Let's create a toy dataset.

In [114]:
multi_turn_dialogs = {
    "Session": [
        # 1. Sending money
        "User: Hi, I need to send money to my cousin in Brazil today.\n"
        "Assistant: Sure! You can transfer up to $3000 USD per day to Brazil.\n"
        "User: I'm sending $2000. Is that okay?\n"
        "Assistant: Yes, that's fine and doesn't require any additional steps.\n"
        "User: Great. The recipient's name is Carla Mendes, CPF number 123.456.789-00. Is that all you need?\n"
        "Assistant: Yes, that's enough to proceed. Please confirm the recipient's bank details on the transfer screen.\n"
        "User: Confirmed, thank you!",

        # 2. Lost card
        "User: I lost my debit card! What do I do?\n"
        "Assistant: Don't worry. You can block your card immediately through the FinBot app.\n"
        "User: How?\n"
        "Assistant: Open the app, go to the Cards section, select the lost card, and tap 'Block card'. It'll be instant.\n"
        "User: Is it reversible?\n"
        "Assistant: No, once blocked, the card can't be unblocked. You'll need to order a replacement.\n"
        "User: Got it. Doing it now. Thanks!",

        # 3. Financial market
        "User: Do you know what the average interest rate is for personal loans in Argentina?\n"
        "Assistant: I'm sorry, I can't provide general financial market information like average interest rates in specific countries.\n"
        "User: Oh, I thought you might have that info.\n"
        "Assistant: I can help with FinBot-specific loan offerings, if available, but I recommend checking with a local financial advisor or bank for up-to-date rates.\n"
        "User: Okay, thanks anyway.",

        # 4. ATM fees
        "User: I'm traveling to Mexico next week. Will I be charged ATM fees there?\n"
        "Assistant: If you use a partner ATM, withdrawals are free. Non-partner ATMs charge $2.50 per withdrawal.\n"
        "User: How do I know which ones are partners?\n"
        "Assistant: You can find a list of partner ATMs in the FinBot app's 'ATM Finder' section.\n"
        "User: Awesome, thanks!",

        # 5. Cancel transaction
        "User: I just sent money to the wrong person! Can I cancel it?\n"
        "Assistant: Unfortunately, once a transaction is processed, it can't be canceled.\n"
        "User: Is there anything I can do?\n"
        "Assistant: You can submit a recall request, but success depends on the recipient's bank cooperation.\n"
        "User: Okay, I'll try that. How do I submit it?\n"
        "Assistant: In the app, go to the transaction details and tap 'Request Recall'. Follow the steps there.\n"
        "User: Got it, thanks for your help."
    ]
}

multi_turn_df = pd.DataFrame(multi_turn_dialogs)

[No output generated]

Run the evaluation:

In [115]:
# Define data definition for Session column (different from earlier definitions)
session_definition = DataDefinition(text_columns=["Session"])

prod_df_3 = Dataset.from_pandas(
    pd.DataFrame(multi_turn_df),
    data_definition=session_definition,
    descriptors=[
        DeclineLLMEval("Session", provider="ollama", model=OLLAMA_LLM_MODEL),
        PIILLMEval("Session", provider="ollama", model=OLLAMA_LLM_MODEL)
    ],
    options=OLLAMA_OPTIONS
)

report = Report([TextEvals()])
my_eval = report.run(prod_df_3)

# ws.add_run(project.id, my_eval, include_data=True)

---------------------------------------------------------------------------
JSONDecodeError                           Traceback (most recent call last)
File /opt/pixi/.pixi/envs/default/lib/python3.13/site-packages/evidently/llm/utils/blocks.py:169, in JsonOutputFormatBlock.parse_response(self, response)
    168 try:
--> 169     return json.loads(response)
    170 except json.JSONDecodeError as e:

File /opt/pixi/.pixi/envs/default/lib/python3.13/json/__init__.py:352, in loads(s, cls, object_hook, parse_float, parse_int, parse_constant, object_pairs_hook, **kw)
    349 if (cls is None and object_hook is None and
    350         parse_int is None and parse_float is None and
    351         parse_constant is None and object_pairs_hook is None and not kw):
--> 352     return _default_decoder.decode(s)
    353 if cls is None:

File /opt/pixi/.pixi/envs/default/lib/python3.13/json/decoder.py:345, in JSONDecoder.decode(self, s, _w)
    341 """Return the Python representation of ``s`` (a ``st

In [116]:
# prod_df_3.as_dataframe()
# my_eval

[No output generated]

In [117]:
raw_dialog_data = prod_df_3.as_dataframe()
raw_dialog_data[(raw_dialog_data["Decline"] == "DECLINE") | (raw_dialog_data["PII"] == "PII")]

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[62], line 1
----> 1 raw_dialog_data = prod_df_3.as_dataframe()
      2 raw_dialog_data[(raw_dialog_data["Decline"] == "DECLINE") | (raw_dialog_data["PII"] == "PII")]

NameError: name 'prod_df_3' is not defined

# Want to play around?

Check docs:
- Descriptor basic API: https://docs.evidentlyai.com/docs/library/descriptors
- Customizing LLM judges: https://docs.evidentlyai.com/metrics/customize_llm_judge
- Custom Python functions: https://docs.evidentlyai.com/metrics/customize_descriptor
- List of all descriptors: https://docs.evidentlyai.com/metrics/all_descriptors

Did you like an example? Star Evidently on GitHub to support the project https://github.com/evidentlyai/evidently

## Shutdown & Cleanup

Run the cell below to unload the Ollama model and shutdown the kernel.

In [ ]:
# === Unload Ollama Model & Shutdown Kernel ===
# Unloads the model from GPU memory before shutting down

try:
    import ollama
    import importlib; importlib.reload(ollama)
    print(f"Unloading Ollama model: {OLLAMA_LLM_MODEL}")
    ollama.generate(model=OLLAMA_LLM_MODEL, prompt="", keep_alive=0)
    print("Model unloaded from GPU memory")
except Exception as e:
    print(f"Model unload skipped: {e}")

# Shut down the kernel to fully release resources
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(restart=False)